# Практическая работа № 2. Работа с LLM через API

**Студент:** Архангельский Данила Андреевич  
**Группа:** ИСТ-05  
**Вариант:** 01 (`rental_01.csv`)


In [ ]:
# При необходимости: %pip install -U langchain-gigachat pandas pydantic python-dotenv
import os, csv
from pathlib import Path
from typing import Optional
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

from google.colab import userdata

API_KEY = userdata.get('GIGA_KEY')
if not API_KEY:
    raise RuntimeError('Добавьте новый GIGA_KEY в локальный .env рядом с ноутбуком')
llm = GigaChat(credentials=API_KEY, model='GigaChat-2', verify_ssl_certs=False, temperature=0.2, max_tokens=1000)
print(llm.invoke('Ответь одним словом: работает?').content)


In [ ]:
VARIANT = '01'
def load_rental(path):
    rows=[]
    with open(path, encoding='utf-8-sig', newline='') as f:
        reader=csv.reader(f, delimiter=';', quotechar='"')
        next(reader, None)
        for row in reader:
            if row: rows.append({'amount': int(row[0]), 'text': ';'.join(row[1:]).strip()})
    return pd.DataFrame(rows)
df = load_rental(Path('data') / f'rental_{VARIANT}.csv').head(15).copy().reset_index(drop=True)
assert len(df)==15
df.head(15)


## Часть 1. Базовый промпт и Few-shot


In [ ]:
basic_prompt = PromptTemplate(input_variables=['text'], template='Определи общее число проживающих, учитывай взрослых и детей, не считай животных. Если данных недостаточно, верни 0. Верни только целое число.\nЗаявка: {text}\nКоличество:')
basic_chain = basic_prompt | llm | StrOutputParser()
system = SystemMessagePromptTemplate.from_template('Извлеки общее число проживающих. Считай взрослых и детей, не считай животных. Примеры: «пара»=2; «один взрослый и ребёнок»=2; «2 взрослых и 2 детей»=4; «с женой и двумя детьми»=4. Если однозначно определить нельзя, верни 0. Только целое число.')
human = HumanMessagePromptTemplate.from_template('Заявка: {text}\nКоличество проживающих:')
few_chain = ChatPromptTemplate.from_messages([system,human]) | llm | StrOutputParser()
def run_chain(chain, texts):
    out=[]
    for text in texts:
        try: out.append(chain.invoke({'text':text}))
        except Exception as e: out.append(f'ERROR: {type(e).__name__}: {e}')
    return out
def to_num(s): return pd.to_numeric(s.astype(str).str.strip(), errors='coerce')
df['result_basic']=run_chain(basic_chain, df.text)
df['result_fewshot']=run_chain(few_chain, df.text)
for col in ['result_basic','result_fewshot']:
    pred=to_num(df[col]); correct=int(pred.eq(df.amount).sum())
    print(f'{col}: {correct}/{len(df)}; ошибок {len(df)-correct}; точность {correct/len(df):.1%}')
errors=df.loc[to_num(df.result_basic).ne(df.amount) | to_num(df.result_fewshot).ne(df.amount), ['text','amount','result_basic','result_fewshot']]
errors


## Выводы по части 1
После запуска зафиксируйте число верных ответов, ошибок и точность для базового промпта и Few-shot. До реального обращения к API численные метрики не рассчитываются.